# Group 6 — Data Ingestion
**ITCS 6190/8190 Cloud Computing for Data Analysis**

This notebook demonstrates data ingestion using Apache Spark Structured APIs.

## Setup Spark Session

In [ ]:
import os, subprocess

JAVA11 = "/usr/local/sdkman/candidates/java/11.0.30-ms"
os.environ["JAVA_HOME"] = JAVA11

# Strip any sdkman/jvm entries from PATH, then prepend Java 11
clean_path = [p for p in os.environ["PATH"].split(":") if "sdkman" not in p and "jvm" not in p]
os.environ["PATH"] = f"{JAVA11}/bin:" + ":".join(clean_path)

v = subprocess.run([f"{JAVA11}/bin/java", "-version"], capture_output=True, text=True)
print(v.stderr.splitlines()[0])  # should print: openjdk version "11..."

Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 34556)
Traceback (most recent call last):
  File "/usr/local/python/3.12.1/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/local/python/3.12.1/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/usr/local/python/3.12.1/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/local/python/3.12.1/lib/python3.12/socketserver.py", line 761, in __init__
    self.handle()
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pyspark/accumulators.py", line 293, in handle
    poll(authenticate_and_accum_updates)
  File "/usr/local/python/3.12.1/lib/python3.12/site-packages/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
   

: 

: 

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp
import os

## Load Raw CSV Files

In [3]:
spark = SparkSession.builder \
    .appName('Group6-Ingestion') \
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", os.environ["AWS_ACCESS_KEY_ID"]) \
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["AWS_SECRET_ACCESS_KEY"]) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED
Picked up JAVA_TOOL_OPTIONS: --add-opens=java.base/javax.security.auth=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED
26/04/20 02:54:27 WARN Utils: Your hostname, codespaces-94354a resolves to a loopback address: 127.0.0.1; using 10.0.12.148 instead (on interface eth0)
26/04/20 02:54:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/python/3.12.1/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/codespace/.ivy2/cache
The jars for the packages stored in: /home/codespace/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5fe361cc-b632-4aa9-b283-ca1b4c5cb895;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 439ms :: artifacts dl 14ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnld

Spark version: 3.5.1


In [4]:
import pyspark, os
jars_path = os.path.join(os.path.dirname(pyspark.__file__), 'jars')
jars = os.listdir(jars_path)
s3_jars = [j for j in jars if 'hadoop-aws' in j or 'aws-java' in j]
print('S3 JARs found:', s3_jars)
print('Jars path:', jars_path)

S3 JARs found: []
Jars path: /usr/local/python/3.12.1/lib/python3.12/site-packages/pyspark/jars


In [5]:
BUCKET = os.environ.get("S3_BUCKET_PATH", "")
if BUCKET:
    customers = spark.read.csv(f'{BUCKET}/raw/customer.csv',     header=True, inferSchema=True)
    products  = spark.read.csv(f'{BUCKET}/raw/product.csv',      header=True, inferSchema=True)
    txns      = spark.read.csv(f'{BUCKET}/raw/transactions.csv',  header=True, inferSchema=True)
    clicks    = spark.read.csv(f'{BUCKET}/raw/click_stream.csv',  header=True, inferSchema=True)
else:
    customers = spark.read.csv('../data/raw/customers.csv',   header=True, inferSchema=True)
    products  = spark.read.csv('../data/raw/products.csv',    header=True, inferSchema=True)
    txns      = spark.read.csv('../data/raw/transactions.csv', header=True, inferSchema=True)
    clicks    = spark.read.csv('../data/raw/click_stream.csv', header=True, inferSchema=True)

print('Customers:   ', customers.count())
print('Products:    ', products.count())
print('Transactions:', txns.count())
print('Clickstream: ', clicks.count())

26/04/20 02:54:51 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Customers:    100000
Products:     44446


Transactions: 852584


Clickstream:  12833602


## Customer Data Trasnforamtion

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType,
    IntegerType, DoubleType, ArrayType, DateType
)

In [7]:
customers_clean = customers \
    .withColumn("birthdate",
        F.to_date("birthdate", "M/d/yyyy")
    ) \
    .withColumn("first_join_date",
        F.to_date("first_join_date", "M/d/yyyy")
    ) \
    .withColumn("gender",
        F.when(F.col("gender") == "F", "Women")
         .when(F.col("gender") == "M", "Men")
         .otherwise("Unisex")
    ) \
    .withColumn("os_type",
        F.when(F.col("device_version").contains("iPhone OS"), "iOS")
         .when(F.col("device_version").contains("Android"),   "Android")
         .when(F.col("device_version").contains("Windows"),   "Windows")
         .otherwise("Other")
    ) \
    .withColumn("os_version",
        F.regexp_replace(
            F.regexp_extract("device_version",
                             r"(?:iPhone OS|Android)\s?([\d_\.]+)", 1),
            "_", "."          
        )
    ) \
    .withColumn("age",
        F.floor(
            F.datediff(F.current_date(), F.col("birthdate")) / 365.25
        ).cast(IntegerType())
    ) \
    .withColumn("tenure_days",
        F.datediff(F.current_date(), F.col("first_join_date"))
    ) \
    .drop("first_name", "last_name", "email", "username",
          "device_id", "device_version")

 
 
    # ── 2a. Parse date strings → proper DateType ──────────
    #   Spark's to_date needs the exact format mask.
    #   "M/d/yyyy" handles both "6/14/1996" and "12/5/2000".

    # ── 2b. Standardize gender to match product table ─────
    #   product.csv uses 'Women'/'Men'/'Unisex'.
    #   Aligning here means JOIN ON gender will work later.

    
    # ── 2c. Extract OS from raw user-agent string ─────────
    #   Raw:   "iPhone; CPU iPhone OS 14_2_1 like Mac OS X"
    #   Clean: os_type="iOS", os_version="14.2.1"

    
    # ── 2d. Derive age (integer years) from birthdate ─────
    #   datediff gives days; dividing by 365.25 handles leap years.
    #   floor() → whole years only (no partial year inflation).

    
    # ── 2e. Bucket age into marketing segments ────────────
    #   Used as a categorical feature in ML models and
    #   as a GROUP BY dimension in SQL segment analysis.

    
    # ── 2f. Customer tenure in days since first join ──────
    #   Key churn signal: longer tenure → lower churn risk.

     
    # ── 2g. Drop PII + columns we replaced ───────────────
    #   first_name, last_name, email, username = PII, not analytical.
    #   device_id, device_version = replaced by os_type / os_version.

print("\ncustomers_clean schema:")
customers_clean.printSchema()
customers_clean.show(3, truncate=False)


customers_clean schema:
root
 |-- customer_id: integer (nullable = true)
 |-- gender: string (nullable = false)
 |-- birthdate: date (nullable = true)
 |-- device_type: string (nullable = true)
 |-- home_location_lat: double (nullable = true)
 |-- home_location_long: double (nullable = true)
 |-- home_location: string (nullable = true)
 |-- home_country: string (nullable = true)
 |-- first_join_date: date (nullable = true)
 |-- os_type: string (nullable = false)
 |-- os_version: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure_days: integer (nullable = true)

+-----------+------+----------+-----------+-------------------+------------------+-------------------+------------+---------------+-------+----------+---+-----------+
|customer_id|gender|birthdate |device_type|home_location_lat  |home_location_long|home_location      |home_country|first_join_date|os_type|os_version|age|tenure_days|
+-----------+------+----------+-----------+-------------------+------------

## Products Data Trasnforamtion 

In [8]:
products_clean = products\
    .withColumnRenamed("id", "product_id") \
    .withColumn("year", F.col("year").cast(IntegerType())) \
    .withColumn("masterCategory", F.trim("masterCategory")) \
    .withColumn("subCategory",    F.trim("subCategory")) \
    .withColumn("articleType",    F.trim("articleType")) \
    .withColumn("baseColour",     F.trim("baseColour")) \
    .withColumn("season",         F.trim("season")) \
    .withColumn("usage",          F.trim("usage"))
 
print("\nProducts_clean schema:")
products_clean.printSchema()
products_clean.show(3, truncate=False)


Products_clean schema:
root
 |-- product_id: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- masterCategory: string (nullable = true)
 |-- subCategory: string (nullable = true)
 |-- articleType: string (nullable = true)
 |-- baseColour: string (nullable = true)
 |-- season: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- usage: string (nullable = true)
 |-- productDisplayName: string (nullable = true)

+----------+------+--------------+-----------+-----------+----------+------+----+------+----------------------------------+
|product_id|gender|masterCategory|subCategory|articleType|baseColour|season|year|usage |productDisplayName                |
+----------+------+--------------+-----------+-----------+----------+------+----+------+----------------------------------+
|15970     |Men   |Apparel       |Topwear    |Shirts     |Navy Blue |Fall  |2011|Casual|Turtle Check Men Navy Blue Shirt  |
|39386     |Men   |Apparel       |Bottomwear |Jeans      

## Transactions Data Transforamtion

In [9]:
txns_ts = txns \
    .withColumn("created_at",
        F.to_timestamp("created_at")) \
    .withColumn("shipment_date_limit",
        F.to_timestamp("shipment_date_limit"))
 
# ── 4b. Numeric casts ─────────────────────────────────────
txns_typed = txns_ts \
    .withColumn("promo_amount",  F.col("promo_amount").cast(DoubleType())) \
    .withColumn("shipment_fee",  F.col("shipment_fee").cast(DoubleType())) \
    .withColumn("total_amount",  F.col("total_amount").cast(DoubleType()))
 
# ── 4c. Sanitize product_metadata ─────────────────────────
#   Single-quote Python dict → double-quote valid JSON string.
#   e.g. [{'product_id': 54728}] → [{"product_id": 54728}]
txns_sanitized = txns_typed.withColumn(
    "product_metadata",
    F.regexp_replace("product_metadata", "'", '"')
)
 
# ── 4d. Define the schema for the JSON array ──────────────
#   Explicit schema is faster than schema inference and
#   guarantees types don't vary row to row.
product_meta_schema = ArrayType(StructType([
    StructField("product_id",  LongType(),    True),
    StructField("quantity",    IntegerType(), True),
    StructField("item_price",  DoubleType(),  True),
]))
 
# ── 4e. Parse JSON → typed column, then explode ──────────
#   from_json turns the string into an actual ArrayType column.
#   explode turns [itemA, itemB] into two separate rows —
#   essential for per-product revenue and recommendation data.
txns_parsed = txns_sanitized.withColumn(
    "products_arr",
    F.from_json("product_metadata", product_meta_schema)
)
 
transactions_exploded = txns_parsed \
    .withColumn("product",    F.explode("products_arr")) \
    .withColumn("product_id", F.col("product.product_id")) \
    .withColumn("quantity",   F.col("product.quantity")) \
    .withColumn("item_price", F.col("product.item_price")) \
    .drop("product_metadata", "products_arr", "product")
 
# ── 4f. Derived columns useful for analysis ───────────────
transactions_exploded = transactions_exploded \
    .withColumn("order_month",
        F.date_format("created_at", "yyyy-MM")
    ) \
    .withColumn("promo_used",
        F.when(
            F.col("promo_code").isNotNull() & (F.col("promo_code") != ""),
            1
        ).otherwise(0)
    ) \
    .withColumn("shipment_days",
        F.datediff(
            F.col("shipment_date_limit").cast(DateType()),
            F.col("created_at").cast(DateType())
        )
    )
 
print("\nTransactions_exploded schema:")
transactions_exploded.printSchema()
transactions_exploded.show(5, truncate=False)


Transactions_exploded schema:
root
 |-- created_at: timestamp (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- booking_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- promo_amount: double (nullable = true)
 |-- promo_code: string (nullable = true)
 |-- shipment_fee: double (nullable = true)
 |-- shipment_date_limit: timestamp (nullable = true)
 |-- shipment_location_lat: double (nullable = true)
 |-- shipment_location_long: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- item_price: double (nullable = true)
 |-- order_month: string (nullable = true)
 |-- promo_used: integer (nullable = false)
 |-- shipment_days: integer (nullable = true)

+--------------------------+-----------+------------------------------------+------------------------------

## Click Stream Data

In [10]:
clicks_ts = clicks.withColumn(
    "event_time", F.to_timestamp("event_time")
)
 
# ── 5b. Sanitize event_metadata ───────────────────────────
clicks_sanitized = clicks_ts.withColumn(
    "event_metadata_clean",
    F.regexp_replace("event_metadata", "'", '"')
)
 
# ── 5c. Union schema covers all event types ───────────────
event_meta_schema = StructType([
    StructField("search_keywords", StringType(), True),  # SEARCH
    StructField("product_id",      LongType(),   True),  # ADD_TO_CART
    StructField("quantity",        IntegerType(),True),  # ADD_TO_CART
    StructField("item_price",      DoubleType(), True),  # ADD_TO_CART
    StructField("promo_code",      StringType(), True),  # ADD_PROMO
    StructField("promo_amount",    DoubleType(), True),  # ADD_PROMO
    StructField("payment_status",  StringType(), True),  # BOOKING
])
 
# ── 5d. Parse metadata → flat columns ────────────────────
#   get each key as its own column so SQL queries can
#   reference them directly without JSON path syntax.
clicks_parsed = clicks_sanitized \
    .withColumn("meta", F.from_json("event_metadata_clean", event_meta_schema)) \
    .withColumn("search_keywords",   F.col("meta.search_keywords")) \
    .withColumn("cs_product_id",     F.col("meta.product_id")) \
    .withColumn("cs_quantity",       F.col("meta.quantity")) \
    .withColumn("cs_item_price",     F.col("meta.item_price")) \
    .withColumn("cs_promo_code",     F.col("meta.promo_code")) \
    .withColumn("cs_promo_amount",   F.col("meta.promo_amount")) \
    .withColumn("cs_payment_status", F.col("meta.payment_status")) \
    .drop("event_metadata", "event_metadata_clean", "meta")
 
clicks_clean = clicks_parsed
 
print("\n✅ clicks_clean schema:")
clicks_clean.printSchema()
clicks_clean.show(5, truncate=False)
 
# ── 5e. Analytical sub-table 1: search_events ─────────────
#   Isolates SEARCH rows with non-null keywords.
#   Used for: trending keyword analysis, search-to-purchase funnel.
search_events = clicks_clean \
    .filter(F.col("event_name") == "SEARCH") \
    .filter(F.col("search_keywords").isNotNull()) \
    .select("session_id", "event_id", "event_time",
            "traffic_source", "search_keywords")
 
print(f"\n✅ search_events : {search_events.count():,} rows")
 
# ── 5f. Analytical sub-table 2: cart_events ───────────────
#   Isolates ADD_TO_CART rows.
#   Used for: cart abandonment detection, item popularity ranking.
cart_events = clicks_clean \
    .filter(F.col("event_name") == "ADD_TO_CART") \
    .select("session_id", "event_id", "event_time", "traffic_source",
            "cs_product_id", "cs_quantity", "cs_item_price") \
    .withColumnRenamed("cs_product_id", "product_id") \
    .withColumnRenamed("cs_quantity",   "quantity") \
    .withColumnRenamed("cs_item_price", "item_price")
 
print(f"✅ cart_events   : {cart_events.count():,} rows")
 
# ── 5g. Analytical sub-table 3: session_funnel ────────────
#   One row per session with binary flags for each funnel stage.
#   This is the PRIMARY FEATURE TABLE for the ML models:
#     • converted = 1/0  → classification target (purchase prediction)
#     • all other cols   → input features (churn + conversion models)
session_funnel = clicks_clean \
    .groupBy("session_id", "traffic_source") \
    .agg(
        F.min("event_time").alias("session_start"),
        F.max("event_time").alias("session_end"),
        F.count("*").alias("total_events"),
        F.max(F.when(F.col("event_name") == "HOMEPAGE",
                     1).otherwise(0)).alias("visited_homepage"),
        F.max(F.when(F.col("event_name") == "SEARCH",
                     1).otherwise(0)).alias("did_search"),
        F.max(F.when(F.col("event_name") == "ADD_TO_CART",
                     1).otherwise(0)).alias("added_to_cart"),
        F.max(F.when(F.col("event_name") == "ADD_PROMO",
                     1).otherwise(0)).alias("used_promo"),
        # converted = 1 if this session ended in a successful BOOKING
        F.max(F.when(
            (F.col("event_name") == "BOOKING") &
            (F.col("cs_payment_status") == "Success"),
            1).otherwise(0)
        ).alias("converted"),
        F.collect_set("search_keywords").alias("keywords_searched"),
    ) \
    .withColumn("session_duration_mins",
        F.round(
            (F.unix_timestamp("session_end") -
             F.unix_timestamp("session_start")) / 60,
            2
        )
    )
 
print(f"✅ session_funnel: {session_funnel.count():,} sessions")
session_funnel.show(3, truncate=False)
 
# ── 5h. Enriched master table: customer_purchases ─────────
#   Join transactions_exploded with customers and products
#   so every purchased item row carries full context.
#   Used for: RFM analysis, customer segmentation, ALS input.
customer_purchases = transactions_exploded \
    .join(
        customers_clean.select(
            "customer_id", "gender", "age",
            "device_type", "os_type", "home_country",
            "tenure_days", "first_join_date"
        ),
        on="customer_id", how="left"
    ) \
    .join(
        products_clean.select(
            "product_id", "masterCategory", "subCategory",
            "articleType", "baseColour", "season", "usage"
        ),
        on="product_id", how="left"
    )
 
print(f"\n✅ customer_purchases: {customer_purchases.count():,} rows")
customer_purchases.show(3, truncate=False)


✅ clicks_clean schema:
root
 |-- session_id: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- event_id: string (nullable = true)
 |-- traffic_source: string (nullable = true)
 |-- search_keywords: string (nullable = true)
 |-- cs_product_id: long (nullable = true)
 |-- cs_quantity: integer (nullable = true)
 |-- cs_item_price: double (nullable = true)
 |-- cs_promo_code: string (nullable = true)
 |-- cs_promo_amount: double (nullable = true)
 |-- cs_payment_status: string (nullable = true)

+------------------------------------+-----------+--------------------------+------------------------------------+--------------+---------------+-------------+-----------+-------------+-------------+---------------+-----------------+
|session_id                          |event_name |event_time                |event_id                            |traffic_source|search_keywords|cs_product_id|cs_quantity|cs_item_price|cs_promo_code|cs


✅ search_events : 1,173,266 rows


✅ cart_events   : 1,937,157 rows


✅ session_funnel: 895,203 sessions


+------------------------------------+--------------+--------------------------+--------------------------+------------+----------------+----------+-------------+----------+---------+-------------------------------------------------------------------------------------------------+---------------------+
|session_id                          |traffic_source|session_start             |session_end               |total_events|visited_homepage|did_search|added_to_cart|used_promo|converted|keywords_searched                                                                                |session_duration_mins|
+------------------------------------+--------------+--------------------------+--------------------------+------------+----------------+----------+-------------+----------+---------+-------------------------------------------------------------------------------------------------+---------------------+
|00003eca-954b-4150-aee1-63fc62f395cf|MOBILE        |2019-08-24 05:47:16.849738|2019-08-


✅ customer_purchases: 1,254,585 rows


26/04/20 03:01:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+-----------+-------------------------+------------------------------------+------------------------------------+--------------+--------------+------------+----------+------------+--------------------------+---------------------+----------------------+------------+--------+----------+-----------+----------+-------------+------+---+-----------+-------+------------+-----------+---------------+--------------+-----------+-----------+----------+------+------+
|product_id|customer_id|created_at               |booking_id                          |session_id                          |payment_method|payment_status|promo_amount|promo_code|shipment_fee|shipment_date_limit       |shipment_location_lat|shipment_location_long|total_amount|quantity|item_price|order_month|promo_used|shipment_days|gender|age|device_type|os_type|home_country|tenure_days|first_join_date|masterCategory|subCategory|articleType|baseColour|season|usage |
+----------+-----------+-------------------------+----------

## Save as Parquet (columnar, 10x compression)

In [11]:
# ============================================================
# 6. REGISTER SPARK SQL TEMP VIEWS
# ============================================================

customers_clean.createOrReplaceTempView("customers")
products_clean.createOrReplaceTempView("products")
transactions_exploded.createOrReplaceTempView("transactions")
clicks_clean.createOrReplaceTempView("clickstream")
search_events.createOrReplaceTempView("search_events")
cart_events.createOrReplaceTempView("cart_events")
session_funnel.createOrReplaceTempView("session_funnel")
customer_purchases.createOrReplaceTempView("customer_purchases")

print("All temp views registered")

# ============================================================
# 7. SAVE ALL PROCESSED TABLES AS PARQUET
# ============================================================

BUCKET = os.environ.get("S3_BUCKET_PATH", "")
PROC   = f"{BUCKET}/processed" if BUCKET else "../data/processed"

TABLES = {
    "customers_clean":       customers_clean,
    "products_clean":        products_clean,
    "transactions_exploded": transactions_exploded,
    "clickstream_clean":     clicks_clean,
    "search_events":         search_events,
    "cart_events":           cart_events,
    "session_funnel":        session_funnel,
    "customer_purchases":    customer_purchases,
}

for name, df in TABLES.items():
    df.write.mode("overwrite").parquet(f"{PROC}/{name}")
    print(f"saved → {PROC}/{name}")

print("\n All tables saved as Parquet")

# ============================================================
# 8. HEALTH-CHECK SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("  StyleIQ Ingestion Pipeline — Health Check")
print("=" * 60)
for name, df in TABLES.items():
    count    = df.count()
    null_sum = df.select([
        F.sum(F.col(c).isNull().cast(IntegerType())).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()
    print(f"  {name:<30} {count:>8,} rows  |  {sum(null_sum.values()):>7,} nulls")
print("=" * 60)
print("\n Pipeline complete")

All temp views registered


saved → s3a://click-stream-s3//processed/customers_clean


saved → s3a://click-stream-s3//processed/products_clean


saved → s3a://click-stream-s3//processed/transactions_exploded


saved → s3a://click-stream-s3//processed/clickstream_clean


...

  clickstream_clean              12,833,602 rows  |  81,344,825 nulls


  search_events                  1,173,266 rows  |        0 nulls


  cart_events                    1,937,157 rows  |        0 nulls


  session_funnel                  895,203 rows  |        0 nulls


  customer_purchases             1,254,585 rows  |  774,074 nulls

 Pipeline complete


## Run the full ingestion script
```bash
python ../src/ingestion.py
```